In [14]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-9f6ffbaf-c954-d8b2-05ff-e1177d373682)


In [3]:
!pip install -q transformers datasets scikit-learn pandas bitsandbytes

import time, gc
import pandas as pd, torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW

# ---- HW4.a settings (unchanged) ----
SEED, SLICE, MAX_LENGTH, LR, EPOCHS, BS = 2733, 733, 128, 2e-5, 3, 32
device = torch.device("cuda")
GPU = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {GPU} | VRAM: {VRAM_GB:.1f} GB")

# ---- Same data + 70/15/15 split as HW4.a ----
df = load_dataset("EleutherAI/twitter-sentiment", split="train").to_pandas()
work_df = df.iloc[SLICE:SLICE + 5000]
train_df, temp_df = train_test_split(work_df, test_size=0.30, random_state=SEED, stratify=work_df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"])

tok = BertTokenizer.from_pretrained("bert-base-uncased")
def make_ds(d):
    enc = tok(d["text"].astype(str).tolist(), padding="max_length", truncation=True,
              max_length=MAX_LENGTH, return_tensors="pt")
    return TensorDataset(enc["input_ids"], enc["attention_mask"],
                         torch.tensor(d["label"].astype(int).tolist()))
train_ds, test_ds = make_ds(train_df), make_ds(test_df)
N = len(train_ds)
print(f"Train: {N} | Test: {len(test_ds)}")

# ---- Helpers ----
def build(ckpt=False, optim="adamw"):
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)
    if ckpt:
        model.gradient_checkpointing_enable()
    if optim == "adamw8bit":
        import bitsandbytes as bnb
        opt = bnb.optim.AdamW8bit(model.parameters(), lr=LR)
    else:
        opt = AdamW(model.parameters(), lr=LR)
    return model, opt

def backward(model, ids, mask, y, amp, accum, scaler):
    with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
        loss = model(input_ids=ids, attention_mask=mask, labels=y).loss / accum
    scaler.scale(loss).backward()

def update(opt, scaler):
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)

def is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()

def max_batch(amp=False, ckpt=False, optim="adamw", cap=8192):
    """Powers of two from 8 until OOM; 2 optimizer steps per size."""
    ok, oom_at, bs = None, None, 8
    g = torch.Generator().manual_seed(SEED)
    while bs <= cap:
        failed, model, opt = False, None, None
        try:
            model, opt = build(ckpt, optim)
            scaler = torch.amp.GradScaler("cuda", enabled=amp)
            model.train()
            for _ in range(2):
                idx = torch.randint(0, N, (bs,), generator=g)
                ids, mask, y = (t[idx].to(device) for t in train_ds.tensors)
                backward(model, ids, mask, y, amp, 1, scaler)
                update(opt, scaler)
            torch.cuda.synchronize()
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if not is_oom(e):
                raise
            failed = True
        model = opt = None
        gc.collect(); torch.cuda.empty_cache()
        if failed:
            oom_at = bs
            break
        ok = bs
        bs *= 2
    return ok, oom_at

def train_eval(amp=False, ckpt=False, accum=1, optim="adamw"):
    """3 epochs at physical batch 32 (HW4 settings), then test accuracy."""
    model, opt = build(ckpt, optim)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    loader = DataLoader(train_ds, batch_size=BS, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model.train(); opt.zero_grad(set_to_none=True)
    opt_steps, pending = 0, 0
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(EPOCHS):
        for ids, mask, y in loader:
            backward(model, ids.to(device), mask.to(device), y.to(device), amp, accum, scaler)
            pending += 1
            if pending == accum:
                update(opt, scaler); opt_steps += 1; pending = 0
        if pending:
            update(opt, scaler); opt_steps += 1; pending = 0
    torch.cuda.synchronize()
    train_time = time.perf_counter() - t0
    peak = torch.cuda.max_memory_allocated() / 1024**3

    model.eval(); correct = 0
    with torch.no_grad():
        for ids, mask, y in DataLoader(test_ds, batch_size=64):
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            correct += (logits.argmax(1).cpu() == y).sum().item()
    model = opt = None
    gc.collect(); torch.cuda.empty_cache()
    return {"train_time_s": train_time,
            "samples_per_sec": EPOCHS * N / train_time,
            "peak_mem_gb": peak,
            "opt_steps": opt_steps,
            "sec_per_opt_step": train_time / opt_steps,
            "test_acc": correct / len(test_ds)}

results = []
CSV = f"partB_{GPU.replace(' ', '_')}.csv"

def run_technique(name, amp=False, ckpt=False, accum=1, optim="adamw"):
    mb, oom = max_batch(amp, ckpt, optim)
    tr = train_eval(amp, ckpt, accum, optim)
    row = {"gpu": GPU, "technique": name, "max_batch": mb, "first_oom": oom,
           "physical_batch": BS, "effective_batch": BS * accum, **tr}
    results.append(row)
    pd.DataFrame(results).to_csv(CSV, index=False)
    print(f"{name}: max batch {mb} (OOM at {oom}) | {tr['samples_per_sec']:.2f} samp/s | "
          f"peak {tr['peak_mem_gb']:.2f} GB | acc {tr['test_acc']:.4f} | "
          f"{tr['opt_steps']} opt steps in {tr['train_time_s']:.1f} s")

print("Setup done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.6 MB/s eta 0:00:00
GPU: Tesla T4 | VRAM: 14.6 GB


README.md:   0%|          | 0.00/501 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  107MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  691kB            

...

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train: 3500 | Test: 750
Setup done.


In [4]:

val_ds = make_ds(val_df)

def train_eval(amp=False, ckpt=False, accum=1, optim="adamw"):
    """Mirrors HW4.a: seed before init and again before training, batch 32, 3 epochs,
    AdamW lr 2e-5, per-epoch validation inside the timed region, test accuracy at end."""
    model, opt = build(ckpt, optim)                              # seeded init (as HW4.a)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)    # re-seed before training (as HW4.a)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BS)
    test_loader = DataLoader(test_ds, batch_size=BS)

    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    opt_steps, pending, train_only = 0, 0, 0.0
    torch.cuda.synchronize(); t_start = time.perf_counter()
    for ep in range(EPOCHS):
        model.train(); opt.zero_grad(set_to_none=True); tl = 0.0
        torch.cuda.synchronize(); t_ep = time.perf_counter()
        for ids, mask, y in train_loader:
            ids, mask, y = ids.to(device), mask.to(device), y.to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                loss = model(input_ids=ids, attention_mask=mask, labels=y).loss
            scaler.scale(loss / accum).backward()
            tl += loss.item()
            pending += 1
            if pending == accum:
                update(opt, scaler); opt_steps += 1; pending = 0
        if pending:
            update(opt, scaler); opt_steps += 1; pending = 0
        torch.cuda.synchronize(); train_only += time.perf_counter() - t_ep

        model.eval(); vl, vc = 0.0, 0
        with torch.no_grad():
            for ids, mask, y in val_loader:
                ids, mask, y = ids.to(device), mask.to(device), y.to(device)
                with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                    out = model(input_ids=ids, attention_mask=mask, labels=y)
                vl += out.loss.item(); vc += (out.logits.argmax(1) == y).sum().item()
        print(f"  Epoch {ep+1}/{EPOCHS} | train loss {tl/len(train_loader):.4f} | "
              f"val loss {vl/len(val_loader):.4f} | val acc {vc/len(val_ds):.4f}")
    torch.cuda.synchronize()
    wall = time.perf_counter() - t_start
    peak = torch.cuda.max_memory_allocated() / 1024**3

    correct = 0
    with torch.no_grad():
        for ids, mask, y in test_loader:
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            correct += (logits.argmax(1).cpu() == y).sum().item()
    model = opt = None
    gc.collect(); torch.cuda.empty_cache()
    return {"wall_time_s": wall,                       # same definition as HW4.a's 221.63 s
            "train_time_s": train_only,                # training steps only
            "samples_per_sec": EPOCHS * N / train_only,
            "peak_mem_gb": peak,
            "opt_steps": opt_steps,
            "sec_per_opt_step": train_only / opt_steps,
            "test_acc": correct / len(test_ds)}

def run_technique(name, amp=False, ckpt=False, accum=1, optim="adamw"):
    print(f"=== {name} ===")
    mb, oom = max_batch(amp, ckpt, optim)
    tr = train_eval(amp, ckpt, accum, optim)
    row = {"gpu": GPU, "technique": name, "max_batch": mb, "first_oom": oom,
           "physical_batch": BS, "effective_batch": BS * accum, **tr}
    results.append(row)
    pd.DataFrame(results).to_csv(CSV, index=False)
    print(f"{name}: max batch {mb} (OOM at {oom}) | {tr['samples_per_sec']:.2f} samp/s | "
          f"peak {tr['peak_mem_gb']:.2f} GB | wall {tr['wall_time_s']:.2f} s | "
          f"test acc {tr['test_acc']:.4f} | {tr['opt_steps']} opt steps")

print("HW4.a-matched training ready.")

HW4.a-matched training ready.


In [5]:
run_technique("Baseline (HW4 settings)")

=== Baseline (HW4 settings) ===


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/3 | train loss 0.5355 | val loss 0.4326 | val acc 0.8200
  Epoch 2/3 | train loss 0.3548 | val loss 0.4159 | val acc 0.8267
  Epoch 3/3 | train loss 0.1986 | val loss 0.5084 | val acc 0.8080
Baseline (HW4 settings): max batch 128 (OOM at 256) | 51.96 samp/s | peak 3.73 GB | wall 217.41 s | test acc 0.7920 | 330 opt steps


In [9]:
from transformers.utils import logging; logging.set_verbosity_error()
run_technique("Mixed precision (AMP fp16)", amp=True)

=== Mixed precision (AMP fp16) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Epoch 1/3 | train loss 0.5376 | val loss 0.4452 | val acc 0.8027
  Epoch 2/3 | train loss 0.3766 | val loss 0.4206 | val acc 0.8253
  Epoch 3/3 | train loss 0.2266 | val loss 0.5152 | val acc 0.8120
Mixed precision (AMP fp16): max batch 256 (OOM at 512) | 160.38 samp/s | peak 3.01 GB | wall 69.65 s | test acc 0.7920 | 330 opt steps


In [7]:
run_technique("Gradient checkpointing", ckpt=True)

=== Gradient checkpointing ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Epoch 1/3 | train loss 0.5358 | val loss 0.4343 | val acc 0.8173
  Epoch 2/3 | train loss 0.3522 | val loss 0.4168 | val acc 0.8147
  Epoch 3/3 | train loss 0.1969 | val loss 0.5110 | val acc 0.8107
Gradient checkpointing: max batch 512 (OOM at 1024) | 35.73 samp/s | peak 2.09 GB | wall 310.14 s | test acc 0.7907 | 330 opt steps


In [8]:
run_technique("Gradient accumulation (32 x 8 = 256)", accum=8)

=== Gradient accumulation (32 x 8 = 256) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Epoch 1/3 | train loss 0.6617 | val loss 0.5977 | val acc 0.6853
  Epoch 2/3 | train loss 0.5370 | val loss 0.4749 | val acc 0.7907
  Epoch 3/3 | train loss 0.4334 | val loss 0.4461 | val acc 0.8133
Gradient accumulation (32 x 8 = 256): max batch 128 (OOM at 256) | 50.65 samp/s | peak 4.17 GB | wall 223.45 s | test acc 0.7973 | 42 opt steps


In [10]:
run_technique("8-bit optimizer (AdamW8bit)", optim="adamw8bit")

=== 8-bit optimizer (AdamW8bit) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Epoch 1/3 | train loss 0.5357 | val loss 0.4343 | val acc 0.8187
  Epoch 2/3 | train loss 0.3535 | val loss 0.4172 | val acc 0.8200
  Epoch 3/3 | train loss 0.2009 | val loss 0.5085 | val acc 0.8133
8-bit optimizer (AdamW8bit): max batch 128 (OOM at 256) | 49.78 samp/s | peak 3.09 GB | wall 227.07 s | test acc 0.7920 | 330 opt steps


In [11]:
run_technique("Best combination (AMP + 8-bit)", amp=True, optim="adamw8bit")

=== Best combination (AMP + 8-bit) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Epoch 1/3 | train loss 0.5352 | val loss 0.4413 | val acc 0.8080
  Epoch 2/3 | train loss 0.3701 | val loss 0.4291 | val acc 0.8120
  Epoch 3/3 | train loss 0.2143 | val loss 0.5027 | val acc 0.8120
Best combination (AMP + 8-bit): max batch 256 (OOM at 512) | 162.17 samp/s | peak 2.40 GB | wall 68.95 s | test acc 0.8027 | 330 opt steps


In [13]:

results = list({r["technique"]: r for r in results}.values())
pd.DataFrame(results).to_csv(CSV, index=False)
pd.DataFrame(results)[["technique", "max_batch", "samples_per_sec", "peak_mem_gb", "test_acc", "opt_steps"]].round(4)

,technique,max_batch,samples_per_sec,peak_mem_gb,test_acc,opt_steps
0,Baseline (HW4 settings),128,51.9591,3.7340,0.7920,330
1,Mixed precision (AMP fp16),256,160.3821,3.0115,0.7920,330
2,Gradient checkpointing,512,35.7321,2.0875,0.7907,330
3,Gradient accumulation (32 x 8 = 256),128,50.6528,4.1666,0.7973,42
4,8-bit optimizer (AdamW8bit),128,49.7790,3.0906,0.7920,330
5,Best combination (AMP + 8-bit),256,162.1697,2.3989,0.8027,330


## Part B — Memory Techniques, Measured One at a Time (Tesla T4, 14.6 GB)

**Setup.** Every run uses the HW4.a fine-tune unchanged except for the one technique being tested: bert-base-uncased, max length 128, AdamW lr 2e-5, physical batch 32, 3 epochs, SEED 2733, rows 733–5732 (70/15/15 split). Accuracy is on the 750-example test set. Peak memory is measured at physical batch 32. Max batch = largest power of two that completes two optimizer steps before out-of-memory. All six runs were on the same Colab T4 session, so they are directly comparable. (Part A reference on an earlier T4 session: max batch 128, 3.73 GB at batch 32; HW4.a test accuracy 0.7960.)

### Table HW4.5.1 — Tesla T4

| Technique | Max batch | Throughput (samp/s) | Peak memory (GB) | Eval accuracy |
|---|---:|---:|---:|---:|
| Baseline (HW4 settings) | 128 | 51.96 | 3.73 | 0.7920 |
| Mixed precision (AMP fp16) | 256 | 160.38 | 3.01 | 0.7920 |
| Gradient checkpointing | 512 | 35.73 | 2.09 | 0.7907 |
| Gradient accumulation (32 × 8) | 128 physical (effective 256) | 50.65 | 4.17 | 0.7973 |
| 8-bit optimiser (AdamW8bit) | 128 | 49.78 | 3.09 | 0.7920 |
| Best combination (AMP + 8-bit) | 256 | 162.17 | 2.40 | 0.8027 |

### 1. Automatic mixed precision
Max batch doubled (128 → 256), throughput rose 3.09× (51.96 → 160.38 samp/s, +208.7%), peak memory fell 0.72 GB (−19.3%), and test accuracy was unchanged (0.7920). The T4's tensor cores run fp16 matrix multiplies much faster than fp32, so AMP saves memory and time together. The memory saving at batch 32 is modest because fp32 master weights and Adam states stay full size; only activations shrink.

### 2. Gradient checkpointing
Max batch rose 4× (128 → 512) and peak memory fell 1.65 GB (−44.1%), the largest saving of any single technique. The cost is compute: throughput dropped from 51.96 to 35.73 samp/s, **−31.2%** (wall-clock +42.7%, 217.4 → 310.1 s), because activations are recomputed during the backward pass. Accuracy was effectively unchanged (0.7907 vs 0.7920, a 1-example difference).

### 3. Gradient accumulation
Physical batch 32 × 8 accumulation steps reached an **effective batch of 256**, a size the T4 cannot run directly (Part A: OOM at 256). Over 3 epochs it took 42 optimiser steps instead of 330. **Time to the same number of optimiser steps (42): 207.3 s with accumulation vs 25.7 s for the baseline** (4.94 s vs 0.61 s per step, 8.1×), because each step processes 8× the data. Throughput per sample was almost unchanged (−2.5%). Peak memory rose 0.43 GB because gradients are held across micro-batches, so accumulation does not save memory; it trades update frequency for effective batch size. Test accuracy was 0.7973; with only 42 updates the model overfit less (val loss still falling at epoch 3: 0.5977 → 0.4749 → 0.4461).

### 4. 8-bit optimiser
AdamW8bit stores the two Adam moment buffers in 8 bits instead of 32, saving 0.64 GB (−17.2%) at a throughput cost of only −4.2% (51.96 → 49.78 samp/s), with no accuracy change (0.7920). Max batch did not change (128): the saving is fixed at roughly the size of the optimiser state, while the next batch doubling needs far more activation memory.

### Best combination: AMP + 8-bit optimiser
Chosen because these were the two techniques that saved memory at little or no throughput cost. Together: peak memory 2.40 GB (−35.8%), throughput 162.17 samp/s (3.12× baseline), max batch 256, and the best test accuracy of all runs (0.8027). Checkpointing was left out because its 31% slowdown is not needed when the model already fits comfortably.

### 5. Ranking by memory saved per unit of throughput lost (batch 32, vs baseline)

| Rank | Technique | Memory saved (GB) | Throughput change (samp/s) | Memory saved per samp/s lost |
|---:|---|---:|---:|---:|
| 1 | Mixed precision | 0.72 | **+108.42** (gain) | No loss — saves memory *and* gains speed |
| 2 | 8-bit optimiser | 0.64 | −2.18 | 0.295 GB |
| 3 | Gradient checkpointing | 1.65 | −16.23 | 0.101 GB |
| 4 | Gradient accumulation | −0.43 (uses more) | −1.31 | Negative — costs memory and throughput |

**Apply first: mixed precision.** It is the only technique with no throughput cost at all: it is 3× faster, halves activation memory, doubles max batch, and does not hurt accuracy.

**Would not bother with on a 32 GB card: gradient checkpointing.** BERT-base at batch 32 needs under 4 GB, and even fp32 batch 128 fits in 11 GB, so a 32 GB card is never memory-limited for this task; paying a 31% slowdown to save memory you don't need is a bad trade. Gradient accumulation is likewise unnecessary there, since batches up to ~256 fit directly.